In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

Data scraping of fantasy pros weekly Fantasy results. This will scrape one week at a time. Most code is deailing with multilevel headers

### Testing out the weekly scraper function

Also printing the results

In [ ]:
def scrape_season_position(
    position: str,
    season: int,
    scoring: str = "ppr",
    weeks = range(1, 19),
    *,
    verbose: bool = False,
    save_dir: str | None = None,
    consolidate: bool = True,
    incremental: bool = True,
    save_per_week: bool = False,
) -> pd.DataFrame:
    """Scrape multiple weeks for a position in a season.

    Args:
        position: {'qb','rb','wr','te','dst'}
        season: Year
        scoring: Scoring type
        weeks: Iterable of week numbers (default 1..18)
        verbose: Diagnostics
        save_dir: If provided, saves consolidated CSV (and optionally per-week CSVs).
        consolidate: When True, returns and optionally saves a single CSV with all weeks.
        incremental: When True, reads existing consolidated CSV and scrapes only missing weeks.
        save_per_week: When True, also save individual week CSVs; default False.

    Returns:
        DataFrame with all successfully scraped weeks (possibly empty).
    """
    import os
    import pandas as pd

    frames = []

    # Determine consolidated path
    consolidated_path = None
    if save_dir and consolidate:
        os.makedirs(save_dir, exist_ok=True)
        consolidated_path = os.path.join(save_dir, f"fantasypros_{position}_{season}_{scoring}.csv")

    existing = pd.DataFrame()
    existing_weeks = set()

    # If incremental, read existing consolidated file and detect already-present weeks
    if incremental and consolidated_path and os.path.exists(consolidated_path):
        try:
            existing = pd.read_csv(consolidated_path)
            if verbose:
                print(f"Loaded existing consolidated: {consolidated_path} rows={len(existing)}")
            if not existing.empty and 'Week' in existing.columns and 'Season' in existing.columns:
                existing_weeks = set(
                    existing[existing['Season'] == season]['Week'].dropna().astype(int).tolist()
                )
                if verbose:
                    print(f"Weeks already present: {sorted(existing_weeks)}")
        except Exception as e:
            if verbose:
                print(f"Failed to read existing consolidated, will recreate: {e}")

    # Filter target weeks to only missing ones
    target_weeks = [int(w) for w in weeks]
    if incremental and existing_weeks:
        target_weeks = [w for w in target_weeks if w not in existing_weeks]
        if verbose:
            print(f"Weeks to scrape: {target_weeks}")

    scraped_weeks = []
    skipped_weeks = []

    # Scrape target weeks
    for w in target_weeks:
        if verbose:
            print(f"Scraping {position.upper()} week {w} {season}")
        per_week_path = None
        if save_dir and save_per_week:
            os.makedirs(save_dir, exist_ok=True)
            per_week_path = os.path.join(save_dir, f"fantasypros_{position}_{season}_wk{w}_{scoring}.csv")
        df_w = scrape_fantasypros_week(
            position,
            season,
            w,
            scoring=scoring,
            verbose=False,
            save_path=per_week_path,
            strict_week=True,
        )
        if df_w.empty:
            skipped_weeks.append(w)
            if verbose:
                print(f"- Skipped week {w}")
            continue
        scraped_weeks.append(w)
        frames.append(df_w)

    # If nothing scraped and we have existing, return existing (for consolidate=True)
    if not frames:
        if verbose:
            print(f"Summary -> scraped: {scraped_weeks}, skipped: {skipped_weeks}")
        if not existing.empty and consolidate:
            return existing
        return pd.DataFrame()

    # Combine newly scraped frames
    df_new = pd.concat(frames, ignore_index=True)

    # Combine with existing, dedupe
    if not existing.empty:
        combined = pd.concat([existing, df_new], ignore_index=True)
    else:
        combined = df_new

    # Deduplicate by Season, Week, Position, and player column if present
    player_col = None
    for cand in ["PLAYER", "Player", "Player Name", "Name"]:
        if cand in combined.columns:
            player_col = cand
            break
    subset_cols = ["Season", "Week", "Position"] + ([player_col] if player_col else [])
    combined = combined.drop_duplicates(subset=subset_cols, keep="last")

    # Save consolidated if requested
    if consolidated_path and consolidate:
        combined.to_csv(consolidated_path, index=False)
        if verbose:
            print(f"Consolidated saved -> {consolidated_path} rows={len(combined)}")
            print(f"Summary -> scraped: {scraped_weeks}, skipped: {skipped_weeks}")

    return combined


TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'